# Phân loại chữ số viết tay MNIST bằng mạng MLP sử dụng PyTorch

Notebook này hướng dẫn bạn cách xây dựng, huấn luyện và đánh giá một mạng **Multi-Layer Perceptron (MLP)** để phân loại chữ số viết tay từ bộ dữ liệu **MNIST** nổi tiếng bằng thư viện **PyTorch**.

### Mục tiêu bài học:
1. Hiểu cách hoạt động của bộ dữ liệu MNIST trong các bài toán Deep Learning.
2. Nắm vững hai khái niệm quan trọng trong PyTorch: **Dataset** và **DataLoader** để xử lý dữ liệu chuyên nghiệp.
3. Cách định nghĩa một kiến trúc mạng MLP kế thừa từ `torch.nn.Module`.
4. Cách viết một **Vòng lặp Huấn luyện (Training Loop)** chuẩn trong thực tế.
5. Cách đánh giá mô hình và trực quan hóa kết quả dự đoán.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập random seed để kết quả đồng nhất qua các lần chạy
torch.manual_seed(42)
np.random.seed(42)

# Kiểm tra thiết bị tính toán (GPU hoặc CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

---
## PHẦN 1: QUẢN LÝ DỮ LIỆU VỚI PYTORCH DATASET & DATALOADER

Trong PyTorch, việc xử lý dữ liệu được tách biệt khỏi mã nguồn huấn luyện mô hình để tăng tính mô-đun và khả năng tái sử dụng. Hai class cốt lõi hỗ trợ việc này là:

1. **`torch.utils.data.Dataset`**:
   * Đại diện cho tập dữ liệu thô (ví dụ: danh sách các đường dẫn ảnh và nhãn, hoặc mảng NumPy).
   * Nhiệm vụ chính: Trả về một mẫu dữ liệu cụ thể kèm nhãn khi ta gọi chỉ số (ví dụ: `dataset[i]`).
   * Hỗ trợ các phép biến đổi ảnh (Data Augmentation, Normalization) thông qua tham số `transform`.

2. **`torch.utils.data.DataLoader`**:
   * Bọc ngoài `Dataset` để cung cấp khả năng duyệt dữ liệu một cách hiệu quả.
   * Nhiệm vụ chính:
     * **Mini-batching**: Tự động gom các mẫu đơn lẻ thành từng batch có kích thước `batch_size` (ví dụ: 64 ảnh một lần).
     * **Shuffling**: Xáo trộn ngẫu nhiên dữ liệu sau mỗi epoch (`shuffle=True`) để tránh mô hình học vẹt theo thứ tự dữ liệu.
     * **Multiprocessing**: Sử dụng đa luồng (`num_workers`) để tải dữ liệu song song giúp tăng tốc độ nạp dữ liệu vào GPU.

### Thiết lập Transforms (Biến đổi dữ liệu)
Trước khi tải dữ liệu, chúng ta cần định nghĩa các phép biến đổi. Ảnh trong bộ dữ liệu MNIST gốc có dạng ảnh xám (PIL Image) kích thước 28x28 pixel với giá trị từ 0 đến 255.
* `transforms.ToTensor()`: Chuyển đổi ảnh PIL hoặc mảng NumPy sang định dạng PyTorch Tensor và tự động chuẩn hóa giá trị pixel về khoảng `[0.0, 1.0]`.
* `transforms.Normalize((0.1307,), (0.3081,))`: Chuẩn hóa dữ liệu về dạng có trung bình (mean) là 0.1307 và độ lệch chuẩn (std) là 0.3081 (đây là các thông số đặc trưng được tính trước trên toàn bộ tập MNIST). Công thức chuẩn hóa:
  $$x_{\text{new}} = \frac{x - \text{mean}}{\text{std}}$$

In [ ]:
# Định nghĩa pipeline biến đổi dữ liệu
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Tải tập dữ liệu MNIST từ torchvision
train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# Khởi tạo DataLoader cho tập Train và tập Test
batch_size = 64

train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=batch_size, 
    shuffle=True
)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=batch_size, 
    shuffle=False
)

print(f"Số lượng mẫu huấn luyện (Train dataset size): {len(train_dataset)}")
print(f"Số lượng mẫu kiểm thử (Test dataset size): {len(test_dataset)}")
print(f"Số lượng batch trong Train Loader: {len(train_loader)} (mỗi batch chứa {batch_size} mẫu)")

### Khảo sát dữ liệu mẫu từ DataLoader
Hãy trích xuất một batch dữ liệu từ `train_loader` để xem kích thước của Tensor ảnh và nhãn, đồng thời trực quan hóa chúng bằng Matplotlib.

In [ ]:
# Trích xuất 1 batch từ DataLoader
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Kích thước tensor batch ảnh: {images.shape} -> (batch_size, channels, height, width)")
print(f"Kích thước tensor batch nhãn: {labels.shape}")

# Trực quan hóa 6 ảnh đầu tiên trong batch
plt.figure(figsize=(10, 4))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    # Ảnh grayscale có 1 channel, cần loại bỏ chiều channel để hiển thị bằng plt.imshow (dùng squeeze)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f"Nhãn: {labels[i].item()}")
    plt.axis('off')
plt.show()

---
## PHẦN 2: ĐỊNH NGHĨA KIẾN TRÚC MẠNG MLP (MULTI-LAYER PERCEPTRON)

Chúng ta sẽ định nghĩa một kiến trúc mạng MLP gồm 3 tầng liên kết đầy đủ (Fully Connected / Dense layers) như sau:
* **Tầng đầu vào (Input Layer)**: 28 x 28 = 784 nơ-ron (chúng ta cần trải phẳng ảnh 2D thành vector 1D).
* **Tầng ẩn 1 (Hidden Layer 1)**: 128 nơ-ron, đi kèm hàm kích hoạt **ReLU**.
* **Tầng ẩn 2 (Hidden Layer 2)**: 64 nơ-ron, đi kèm hàm kích hoạt **ReLU**.
* **Tầng đầu ra (Output Layer)**: 10 nơ-ron (tương ứng với 10 lớp chữ số từ 0 đến 9).

### Lưu ý về hàm kích hoạt ở tầng đầu ra:
Trong PyTorch, khi sử dụng hàm mất mát `nn.CrossEntropyLoss()`, hàm này đã **tự động** tích hợp bước kích hoạt Softmax bên trong nó (chính xác là LogSoftmax + NLLLoss). Do đó, tầng cuối cùng của mô hình không cần áp dụng thêm hàm kích hoạt Softmax, mà chỉ cần trả về các giá trị tuyến tính thô (được gọi là **logits**).

In [ ]:
class MNIST_MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim1=128, hidden_dim2=64, output_dim=10):
        super(MNIST_MLP, self).__init__()
        
        # Định nghĩa các tầng tuyến tính (Fully Connected / Linear)
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        
        # Hàm kích hoạt ReLU
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Trải phẳng ảnh từ (batch_size, 1, 28, 28) thành (batch_size, 784)
        x = x.view(x.size(0), -1)
        
        # Tầng ẩn 1
        out = self.fc1(x)
        out = self.relu(out)
        
        # Tầng ẩn 2
        out = self.fc2(out)
        out = self.relu(out)
        
        # Tầng đầu ra (Logits)
        logits = self.fc3(out)
        return logits

# Khởi tạo mô hình và chuyển sang thiết bị tính toán (CPU/GPU)
model = MNIST_MLP().to(device)
print(model)

### Định nghĩa Hàm mất mát (Loss Function) và Bộ tối ưu hóa (Optimizer)
* **Loss Function**: Sử dụng `nn.CrossEntropyLoss()` rất phù hợp cho bài toán phân loại đa lớp.
* **Optimizer**: Sử dụng thuật toán **Adam** (`optim.Adam`), là một thuật toán tối ưu hóa tự điều chỉnh tốc độ học (adaptive learning rate) giúp hội tụ nhanh hơn nhiều so với SGD truyền thống.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

---
## PHẦN 3: VIẾT VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP)

Vòng lặp huấn luyện là nơi diễn ra thuật toán **Lan truyền tiến (Forward)**, **Tính toán Loss**, **Lan truyền ngược (Backward)** để tính gradients và **Cập nhật tham số**.

Một epoch huấn luyện chuẩn sẽ chạy qua các bước:
1. Đặt mô hình ở chế độ huấn luyện: `model.train()`.
2. Duyệt qua từng batch dữ liệu từ `train_loader`.
3. Xóa các gradient đã tính ở bước trước bằng `optimizer.zero_grad()`.
4. Thực hiện forward pass để lấy dự đoán: `outputs = model(images)`.
5. Tính toán Loss bằng cách so sánh `outputs` với nhãn thực tế `labels`.
6. Tính toán gradient bằng lan truyền ngược: `loss.backward()`.
7. Cập nhật các tham số (weights & biases) bằng `optimizer.step()`.

In [ ]:
epochs = 10
train_loss_history = []
train_acc_history = []

print("=== BẮT ĐẦU HUẤN LUYỆN ===")
for epoch in range(epochs):
    model.train() # Đặt mô hình ở chế độ huấn luyện
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        # Chuyển dữ liệu lên GPU nếu khả dụng
        images, labels = images.to(device), labels.to(device)
        
        # 1. Xóa sạch các gradient cũ
        optimizer.zero_grad()
        
        # 2. Forward pass: Dự đoán đầu ra
        outputs = model(images)
        
        # 3. Tính Loss
        loss = criterion(outputs, labels)
        
        # 4. Backward pass: Lan truyền ngược tính gradient
        loss.backward()
        
        # 5. Cập nhật trọng số
        optimizer.step()
        
        # Thống kê kết quả huấn luyện của batch hiện tại
        running_loss += loss.item() * images.size(0)
        
        # outputs có shape: (batch_size, 10). Giá trị lớn nhất trong mỗi hàng chính là lớp dự đoán
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / total
    
    train_loss_history.append(epoch_loss)
    train_acc_history.append(epoch_acc)
    
    print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2%}")
print("=== HUẤN LUYỆN HOÀN TẤT ===")

---
## PHẦN 4: ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP KIỂM THỬ (TEST EVALUATION)

Để đánh giá chính xác khả năng tổng quát hóa của mô hình trên dữ liệu mới chưa từng thấy, chúng ta kiểm tra mô hình trên tập `test_loader`.

### Hai lưu ý quan trọng khi Evaluate:
1. **`model.eval()`**: Chuyển mô hình sang chế độ đánh giá. Việc này sẽ tắt một số lớp đặc thù như Dropout hoặc Batch Normalization (nếu có) để đảm bảo mô hình dự đoán nhất quán.
2. **`torch.no_grad()`**: Tắt chế độ tính đạo hàm tự động (autograd). Khi đánh giá, chúng ta không cần cập nhật trọng số, việc tắt autograd giúp tiết kiệm dung lượng bộ nhớ GPU/RAM và tăng tốc độ xử lý lên rất nhiều.

In [ ]:
model.eval() # Đặt mô hình ở chế độ đánh giá
test_correct = 0
test_total = 0
test_loss = 0.0

with torch.no_grad(): # Tắt tính toán gradient
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

final_test_loss = test_loss / len(test_dataset)
final_test_acc = test_correct / test_total

print(f"=== ĐÁNH GIÁ TRÊN TẬP TEST ===")
print(f"Test Loss: {final_test_loss:.4f}")
print(f"Test Accuracy: {final_test_acc:.2%}")

---
## PHẦN 5: TRỰC QUAN HÓA KẾT QUẢ VÀ DỰ ĐOÁN THỰC TẾ

Chúng ta tiến hành vẽ đồ thị lịch sử huấn luyện (Loss Curve) và hiển thị kết quả dự đoán trực quan trên một số mẫu ảnh kiểm thử.

In [ ]:
# 1. Vẽ đồ thị Lịch sử Loss & Accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_loss_history, label='Train Loss', color='tab:red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Đồ thị Loss qua các Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_acc_history, label='Train Accuracy', color='tab:blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Đồ thị Accuracy qua các Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# 2. Hiển thị dự đoán thực tế trên một vài ảnh test
images, labels = next(iter(test_loader))
images_device, labels_device = images.to(device), labels.to(device)

# Lấy dự đoán từ mô hình
model.eval()
with torch.no_grad():
    outputs = model(images_device)
    _, preds = torch.max(outputs, 1)

# Chuyển dữ liệu về CPU để hiển thị bằng Matplotlib
images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    
    # Hiển thị màu xanh nếu dự đoán đúng, màu đỏ nếu dự đoán sai
    color = 'green' if preds[i] == labels[i] else 'red'
    plt.title(f"Thực tế: {labels[i]}\nDự đoán: {preds[i]}", color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()